# Customer Segmentation using RFM & K-Means

This notebook segments **2,000 customers** using Recency, Frequency, and Monetary (RFM) behavior.

### Workflow
1. Load and validate the dataset
2. Standardize RFM features
3. Evaluate candidate values of K
4. Build the final four-cluster K-Means solution
5. Profile and label customer segments
6. Visualize the five most decision-relevant findings
7. Export the segmented dataset


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
SEGMENT_ORDER = ["VIP", "Regular", "Occasional", "At Risk"]


## 1. Load Data


In [ ]:
df = pd.read_excel("RFM_KMeans_2000_Customers.xlsx")
print("Dataset shape:", df.shape)
display(df.head())


## 2. Data Quality Checks


In [ ]:
required = ["CustomerID", "Recency", "Frequency", "Monetary"]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

display(df[required].isna().sum().to_frame("Missing Values"))
print("Duplicate Customer IDs:", df["CustomerID"].duplicated().sum())
display(df[["Recency","Frequency","Monetary"]].describe().round(2))


## 3. Prepare and Scale RFM Features


In [ ]:
rfm_features = ["Recency", "Frequency", "Monetary"]
X = df[rfm_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 4. Evaluate the Number of Clusters

The Elbow Method and Silhouette Score are used to make the cluster choice transparent.  
For this business segmentation project, **K = 4** is retained to produce four interpretable customer groups.


In [ ]:
evaluation = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X_scaled)
    evaluation.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette_score": silhouette_score(X_scaled, labels)
    })

evaluation_df = pd.DataFrame(evaluation)
display(evaluation_df.round(4))

plt.figure(figsize=(8,5))
plt.plot(evaluation_df["k"], evaluation_df["inertia"], marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.show()

plt.figure(figsize=(8,5))
plt.plot(evaluation_df["k"], evaluation_df["silhouette_score"], marker="o")
plt.title("Silhouette Score by K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.show()


## 5. Build the Final K-Means Model


In [ ]:
model = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=20)
df["Cluster"] = model.fit_predict(X_scaled)

cluster_means = df.groupby("Cluster")[rfm_features].mean()
display(cluster_means.round(2))


## 6. Translate Clusters into Business Segments

Cluster IDs are arbitrary, so segment names are assigned from the observed RFM profiles rather than hard-coded cluster numbers.


In [ ]:
means = df.groupby("Cluster")[rfm_features].mean()

vip_cluster = means["Monetary"].idxmax()
remaining = [c for c in means.index if c != vip_cluster]

at_risk_cluster = means.loc[remaining, "Recency"].idxmax()
remaining = [c for c in remaining if c != at_risk_cluster]

regular_cluster = means.loc[remaining, "Frequency"].idxmax()
occasional_cluster = [c for c in remaining if c != regular_cluster][0]

cluster_names = {
    vip_cluster: "VIP",
    regular_cluster: "Regular",
    occasional_cluster: "Occasional",
    at_risk_cluster: "At Risk"
}

df["Segment"] = df["Cluster"].map(cluster_names)

segment_profile = df.groupby("Segment").agg(
    Customers=("CustomerID","count"),
    Avg_Recency=("Recency","mean"),
    Avg_Frequency=("Frequency","mean"),
    Avg_Monetary=("Monetary","mean"),
    Total_Monetary=("Monetary","sum")
).reindex(SEGMENT_ORDER)

display(segment_profile.round(2))


# 7. Key Visual Analysis

The following five visuals were selected because together they explain **segment size, customer value, purchasing intensity, recency behavior, and the overall RFM profile**.


### Visual 1 — Customer Distribution by Segment

Shows the size of each customer segment and highlights where most customers are concentrated.


In [ ]:
plt.figure(figsize=(8,5))
segment_counts = df["Segment"].value_counts().reindex(SEGMENT_ORDER)
segment_counts.plot(kind="bar")
plt.title("Customer Distribution by Segment")
plt.xlabel("Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Visual 2 — Average Monetary Value by Segment

Compares the average customer value across segments and makes the high-value groups immediately visible.


In [ ]:
plt.figure(figsize=(8,5))
segment_profile["Avg_Monetary"].plot(kind="bar")
plt.title("Average Monetary Value by Segment")
plt.xlabel("Segment")
plt.ylabel("Average Monetary")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Visual 3 — Frequency vs Monetary

Shows the relationship between purchase frequency and customer value, colored by business segment.


In [ ]:
plt.figure(figsize=(9,6))
for segment in SEGMENT_ORDER:
    subset = df[df["Segment"] == segment]
    plt.scatter(
        subset["Frequency"],
        subset["Monetary"],
        alpha=0.55,
        label=segment
    )

plt.title("Frequency vs Monetary by Customer Segment")
plt.xlabel("Frequency")
plt.ylabel("Monetary")
plt.legend()
plt.tight_layout()
plt.show()


### Visual 4 — Recency vs Frequency

Separates active/high-frequency customers from customers whose last purchase was much longer ago.


In [ ]:
plt.figure(figsize=(9,6))
for segment in SEGMENT_ORDER:
    subset = df[df["Segment"] == segment]
    plt.scatter(
        subset["Recency"],
        subset["Frequency"],
        alpha=0.55,
        label=segment
    )

plt.title("Recency vs Frequency by Customer Segment")
plt.xlabel("Recency")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


### Visual 5 — RFM Segment Profile

Provides a compact comparison of the relative Recency, Frequency, and Monetary characteristics of all four segments.


In [ ]:
profile = segment_profile[["Avg_Recency","Avg_Frequency","Avg_Monetary"]].copy()

normalized_profile = (
    (profile - profile.min()) /
    (profile.max() - profile.min())
)

plt.figure(figsize=(8,5))
plt.imshow(normalized_profile.values, aspect="auto")
plt.colorbar(label="Normalized Value")
plt.xticks(range(3), ["Recency","Frequency","Monetary"])
plt.yticks(range(len(SEGMENT_ORDER)), SEGMENT_ORDER)
plt.title("RFM Segment Profile")

for i in range(normalized_profile.shape[0]):
    for j in range(normalized_profile.shape[1]):
        plt.text(
            j, i,
            f"{normalized_profile.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()


## 8. Business Interpretation

- **VIP:** highest-value customers; prioritize retention and loyalty.
- **Regular:** engaged repeat customers with potential to move into VIP.
- **Occasional:** largest lower-engagement group; focus on increasing purchase frequency and value.
- **At Risk:** customers with the longest recency; prioritize reactivation campaigns.


## 9. Export Final Segmented Dataset


In [ ]:
output_file = "RFM_KMeans_2000_Customers_Segmented.xlsx"
df.to_excel(output_file, index=False)
print(f"Saved successfully: {output_file}")
